In [1]:
9+4

13

In [2]:
from dotenv import load_dotenv
from IPython.display import Image, display
from pydantic import BaseModel,Field
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langchain_mcp_adapters.client import MultiServerMCPClient
load_dotenv(override=True)

True

In [7]:
agent = create_agent(
    model="openai:gpt-3.5-turbo",
    system_prompt='You are a helpful assistant who answer concisely'
)

'''result=agent.invoke({"messages":[{"role":"user","content":"what is Model context protocol"}]})
print(result["messages"][-1].content)'''

result = await agent.ainvoke({"messages":[{"role":"user","content":"why Async code suit agents so well"}]})
print(result['messages'][-1].content)

Async code is well-suited for agents because it allows them to perform multiple tasks concurrently without blocking the main thread. This means that agents can efficiently handle multiple requests, process data in the background, and respond to events without slowing down the overall application.


In [ ]:


result=agent.invoke({'messages':[{'role':'user',"content":'what is the weather in London and it population'}]})
print(result['messages'][-1].content)

The population of London is 8.9 million and the weather in London is currently rainy with a temperature of 14 degrees.


In [ ]:
from dotenv import load_dotenv
from IPython.display import Image, display
from pydantic import BaseModel,Field
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langchain_mcp_adapters.client import MultiServerMCPClient
load_dotenv(override=True)


@tool
def get_weather(city: str) -> str:
    """Return today's weather for a specified city."""
    pretend = {"London": "rainy, 14 degrees","Rome": "sunny, 27 degrees"}
    return pretend.get(city.title(), "unknown")

memory_agent= create_agent(model="openai:gpt-3.5-turbo",tools=[get_weather],checkpointer=MemorySaver(),
system_prompt="You are a travel assistant."
        "Use your tools to answer questions about cities. "
        "Remember the destination city mentioned earlier in the conversation. "
        "When the user asks about the weather for their destination, "
        "call get_weather using that destination city."
        )
config = {'configurable':{'thread_id':'trip-1'}}
memory_agent.invoke({'messages':[{'role':'user','content':'what is weather in Rome'}]},config=config)
result = memory_agent.invoke({'messages':[{'role':'user','content':'what is the weather where i am going to my trip'}]},config=config)
print(result['messages'][-1].content)

The weather in Rome, your destination city, is currently sunny with a temperature of 27 degrees.


In [32]:
## Part 4: structured output

class CityReport(BaseModel):
    city: str = Field(description="The city name")
    weather:str = Field(description="A short weather description")
    population: str = Field(description="the population")

report_agent = create_agent(
    model="openai:gpt-3.5-turbo",
    tools=[get_weather],
    response_format=CityReport
)

result= report_agent.invoke({'messages':[{'role':"user","content":"Give me a report on London"}]})
report= result["structured_response"]
print(result)
print(report.weather)

{'messages': [HumanMessage(content='Give me a report on London', additional_kwargs={}, response_metadata={}, id='5f056db7-f378-429e-ba72-09fb12e6da9f'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 91, 'total_tokens': 105, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EbNUUhy0RcfwSnMwalVAx5jt24Nx3', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a15dec-7b51-73b5-8e86-0cb0d3c98039-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'London'}, 'id': 'call_nwdm57E1PXAxVffzJCMVeR61', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 9

In [34]:
@wrap_tool_call
def log_tool_calls(request, handler):
    call = request.tool_call
    print(f"[middleware] calling {call['name']} with {call['args']}")
    return handler(request)
watched_agent= create_agent(model='openai:gpt-3.5-turbo',tools=[get_weather],
                            system_prompt="You are a travel agent assistant, use you tools",
                            middleware=[log_tool_calls])

result=watched_agent.invoke({"messages":[{'role':"user","content":'weather of London and Rome'}]})
print(result['messages'][-1].content)

[middleware] calling get_weather with {'city': 'London'}
[middleware] calling get_weather with {'city': 'Rome'}
The weather in London is rainy with 14 degrees, and in Rome, it's sunny with 27 degrees.


In [35]:
!npx -y playwright@latest screenshot --channel=chrome https://news.ycombinator.com playwright_check.png
display(Image("playwright_check.png"))

'npx' is not recognized as an internal or external command,
operable program or batch file.


FileNotFoundError: No such file or directory: 'playwright_check.png'

FileNotFoundError: No such file or directory: 'playwright_check.png'

<IPython.core.display.Image object>